# Data Structures Master Notebook
Run the code blocks below to experiment with the various data structure implementations in Python.

## Arrays / Dynamic Arrays

<strong>ADT:</strong> indexed sequence supporting O(1) positional access. <br><strong>Impl:</strong> contiguous block of fixed-size memory slots; Python's <code>list</code> is a dynamic array — an over-allocated raw array of <em>pointers</em> to objects, resized via geometric growth when capacity is exceeded.

<pre class="diagram">
index:      0     1     2     3     4    (len=5, capacity=8)
         +-----+-----+-----+-----+-----+-----+-----+-----+
buffer:  | ptr | ptr | ptr | ptr | ptr |  .  |  .  |  .  |
         +--|--+--|--+--|--+--|--+--|--+-----+-----+-----+
            v     v     v     v     v
           10    20    30    40    50    <- heap objects

Append past capacity -> allocate new buffer (~1.125x growth),
copy all pointers, free old buffer.  O(n) worst, O(1) amortized.</pre>
<table>
<tr><th>Op</th><th>Method</th><th>Best</th><th>Avg</th><th>Worst</th><th>Space</th></tr>
<tr><td>Access</td><td>a[i]</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Search</td><td>x in a</td><td>O(1)</td><td>O(n)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Append</td><td>.append(x)</td><td>O(1)</td><td>O(1)*</td><td>O(n)</td><td>O(1)*</td></tr>
<tr><td>Insert</td><td>.insert(i,x)</td><td>O(1)</td><td>O(n)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Delete</td><td>.pop(i)</td><td>O(1)</td><td>O(n)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Pop end</td><td>.pop()</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Slice</td><td>a[i:j]</td><td>O(k)</td><td>O(k)</td><td>O(k)</td><td>O(k)</td></tr>
</table>
<div style="font-size:0.75rem; color:var(--text-secondary); margin-bottom: 0.5rem;">*amortized (geometric resize)</div>
<strong>PITFALLS</strong>
<ul class="pitfalls">
<li><code>a[i:j]=x</code> and <code>del/insert(0,x)</code> shift all elements — O(n), silent perf trap in loops.</li>
<li>Shallow copy: <code>b = a</code> aliases; use <code>a.copy()</code> / <code>a[:]</code> to avoid mutating shared refs.</li>
<li><code>[[0]*n]*m</code> creates m references to <em>the same</em> inner list — classic 2D-array bug.</li>
<li>Over-allocation trades memory for amortized O(1) append; <code>sys.getsizeof</code> > sum of elements.</li>
</ul>

In [ ]:
arr = [10, 20, 30, 40, 50]
arr2 = list(range(5))
import array
typed = array.array('i', [1, 2, 3])  # packed C ints, no pointer overhead

print('Array:', arr)
print('Typed array:', typed)

## Singly & Doubly Linked Lists

<strong>ADT:</strong> ordered sequence with O(1) insert/delete given a node reference. <br><strong>Impl:</strong> non-contiguous heap nodes linked via pointers — no built-in Python type; roll your own or use <code>collections.deque</code> (doubly-linked block list).

<pre class="diagram">
Singly:
head                                       tail
 |                                          |
 v                                          v
[10|next]->[20|next]->[30|next]->[40|None]

Doubly:
head                                       tail
 |                                          |
 v                                          v
None<-[10|n|p]<->[20|n|p]<->[30|n|p]->[40|n|p]->None
      prev<-     <-prev     prev->     prev-></pre>
<table>
<tr><th>Op</th><th>Method</th><th>Best</th><th>Avg</th><th>Worst</th><th>Space</th></tr>
<tr><td>Access(i)</td><td>traverse</td><td>O(1)</td><td>O(n)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Push front</td><td>push_front</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Push back</td><td>push_back</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Insert after node</td><td>-</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Delete (given node)</td><td>-</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Search</td><td>-</td><td>O(1)</td><td>O(n)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>deque append/pop</td><td>deque.append</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
</table>
<strong>PITFALLS</strong>
<ul class="pitfalls">
<li>Forgetting to update <strong>both</strong> prev & next pointers on delete/insert corrupts the chain silently.</li>
<li>Losing the only reference to head/tail during traversal leaks the remainder of the list (GC-collected but logically lost).</li>
<li>No O(1) random access — never use a linked list when frequent indexing is required.</li>
<li>Per-node pointer + object overhead (~56+ bytes/node in CPython) >> array's packed layout.</li>
</ul>

In [ ]:
class Node:
    __slots__ = ('val', 'next', 'prev')
    def __init__(self, val, next=None, prev=None):
        self.val, self.next, self.prev = val, next, prev

class DoublyLinkedList:
    def __init__(self):
        self.head = self.tail = None
        self.size = 0
    
    def push_front(self, val):
        n = Node(val, next=self.head)
        if self.head: self.head.prev = n
        self.head = n
        self.tail = self.tail or n
        self.size += 1

    def print_list(self):
        curr = self.head
        while curr:
            print(curr.val, end=' <-> ')
            curr = curr.next
        print('None')

dll = DoublyLinkedList()
dll.push_front(30)
dll.push_front(20)
dll.push_front(10)
dll.print_list()

## Stacks & Queues

<strong>ADT:</strong> Stack = LIFO; Queue = FIFO — both restrict access to ends. <br><strong>Impl:</strong> Stack via dynamic array (<code>list</code>); Queue via <code>collections.deque</code> (doubly-linked array blocks) — <strong>never</strong> use list as a queue (O(n) pop(0)).

<pre class="diagram">
Stack (LIFO)             Queue (FIFO)
push -> [ 30 ] top       enqueue->[10|20|30]->dequeue
        [ 20 ]                    front ^    ^ rear
        [ 10 ] <- bottom
pop <-  top</pre>
<table>
<tr><th>Op</th><th>Method</th><th>Best</th><th>Avg</th><th>Worst</th><th>Space</th></tr>
<tr><td>Push (stack)</td><td>list.append</td><td>O(1)</td><td>O(1)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Pop (stack)</td><td>list.pop()</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Peek</td><td>list[-1]</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Enqueue</td><td>deque.append</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Dequeue</td><td>deque.popleft</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>list.pop(0)</td><td>list.pop(0)</td><td>O(n)</td><td>O(n)</td><td>O(n)</td><td>O(1)</td></tr>
</table>
<strong>PITFALLS</strong>
<ul class="pitfalls">
<li><code>list.pop(0)</code> / <code>list.insert(0,x)</code> for a queue is O(n) — silently degrades to quadratic in loops.</li>
<li>Unbounded recursion mimics an implicit call stack — can blow Python's recursion limit (~1000) where an explicit stack wouldn't.</li>
<li>Stack overflow (deep recursion) vs. queue starvation (priority queues) are distinct failure modes — don't conflate.</li>
<li><code>deque(maxlen=n)</code> silently discards elements from the opposite end when full — a sneaky data-loss trap.</li>
</ul>

In [ ]:
stack = []
stack.append(1)
stack.append(2)
print('Stack after appends:', stack)
print('Popped from stack:', stack.pop())

from collections import deque
queue = deque()
queue.append(1)
queue.append(2)
print('Queue after appends:', queue)
print('Popped from queue:', queue.popleft())

## Hash Tables (dict / set)

<strong>ADT:</strong> associative array mapping keys to values with expected O(1) access. <br><strong>Impl:</strong> array of buckets indexed by <code>hash(key) % capacity</code>; CPython dict uses open addressing + a compact dense array for insertion-ordered storage since 3.7.

<pre class="diagram">
hash("cat")=17 -> slot 17 % 8 = 1
     slot:  0      1      2      3    ...
   bucket: [ - ] [cat:5] [ - ] [dog:2] ...
Collision (probing): slot occupied -> perturb & probe next slot
                     (open addressing, not chaining, in CPython)</pre>
<table>
<tr><th>Op</th><th>Method</th><th>Best</th><th>Avg</th><th>Worst</th><th>Space</th></tr>
<tr><td>Insert</td><td>d[k]=v</td><td>O(1)</td><td>O(1)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Lookup</td><td>d[k] / k in d</td><td>O(1)</td><td>O(1)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Delete</td><td>del d[k]</td><td>O(1)</td><td>O(1)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Iterate</td><td>for k in d</td><td>O(n)</td><td>O(n)</td><td>O(n)</td><td>O(1)</td></tr>
</table>
<strong>PITFALLS</strong>
<ul class="pitfalls">
<li>Mutable keys (lists) are unhashable by design — mutating an object used as a key after insertion corrupts lookups.</li>
<li>Worst case O(n) under pathological/adversarial hash collisions (mitigated by PYTHONHASHSEED randomization).</li>
<li>Resizing (rehashing all keys) triggers on load factor threshold — occasional O(n) latency spikes.</li>
<li>Dict preserves insertion order (3.7+) but this is <em>not</em> the same guarantee as a sorted structure — don't rely on it for ordering logic.</li>
</ul>

In [ ]:
d = {"cat": 5, "dog": 2}
d["fox"] = 9
s = {1, 2, 3}

from collections import defaultdict, Counter
dd = defaultdict(list)
dd['animals'].append('cat')

cnt = Counter("mississippi")

print('Dict:', d)
print('Set:', s)
print('DefaultDict:', dd)
print('Counter:', cnt)

## Binary Search Trees & AVL Trees

<strong>ADT:</strong> ordered hierarchical set/map maintaining left < node < right invariant. <br><strong>Impl:</strong> linked nodes with left/right pointers; AVL adds a self-balancing invariant (|height(L)-height(R)| ≤ 1) via rotations, bounding height to O(log n).

<pre class="diagram">
       BST                   AVL after rotation (LL case)
       (50)                         (30)
      /    \                       /       (30)    (70)     -->         (20)    (50)
   /  \                            \       /  (20) (40)                          (40)  (70)
 /
(10)   <- unbalanced! height=4       balanced, height=2
       left-heavy, O(n) worst</pre>
<table>
<tr><th>Op</th><th>Struct</th><th>Best</th><th>Avg</th><th>Worst</th><th>Space</th></tr>
<tr><td>Search</td><td>BST</td><td>O(1)</td><td>O(log n)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Insert</td><td>BST</td><td>O(1)</td><td>O(log n)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Delete</td><td>BST</td><td>O(1)</td><td>O(log n)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Search</td><td>AVL</td><td>O(1)</td><td>O(log n)</td><td>O(log n)</td><td>O(1)</td></tr>
<tr><td>Insert</td><td>AVL</td><td>O(log n)</td><td>O(log n)</td><td>O(log n)</td><td>O(1)</td></tr>
<tr><td>Delete</td><td>AVL</td><td>O(log n)</td><td>O(log n)</td><td>O(log n)</td><td>O(1)</td></tr>
<tr><td>In-order traversal</td><td>both</td><td>O(n)</td><td>O(n)</td><td>O(n)</td><td>O(h)</td></tr>
</table>
<strong>PITFALLS</strong>
<ul class="pitfalls">
<li>Inserting sorted/near-sorted data into a plain BST degenerates it into a linked list — O(n) ops, defeats the point.</li>
<li>Deletion with two children requires the in-order successor/predecessor swap — easy to mishandle subtree reattachment.</li>
<li>AVL rebalancing (4 rotation cases: LL/RR/LR/RL) is easy to get subtly wrong — always update heights bottom-up first.</li>
<li>Recursive implementations risk stack overflow on very deep/degenerate trees; consider iterative or increase recursion limit.</li>
</ul>

In [ ]:
class BSTNode:
    __slots__ = ('key', 'left', 'right', 'height')
    def __init__(self, key):
        self.key, self.left, self.right, self.height = key, None, None, 1

def insert(root, key):
    if not root: return BSTNode(key)
    if key < root.key: 
        root.left = insert(root.left, key)
    elif key > root.key: 
        root.right = insert(root.right, key)
    return root

def inorder(root):
    if root:
        inorder(root.left)
        print(root.key, end=' ')
        inorder(root.right)

root = None
for k in [50, 30, 70, 20, 40]:
    root = insert(root, k)

print('Inorder traversal:')
inorder(root)
print()

## Heaps / Priority Queues

<strong>ADT:</strong> collection supporting O(log n) insert and O(log n) extract-min/max with O(1) peek. <br><strong>Impl:</strong> complete binary tree stored implicitly in a flat array — <code>heapq</code> is a binary min-heap over a plain Python list.

<pre class="diagram">
Min-heap tree:            Array repr (0-indexed):
     (5)                      idx:  0  1  2  3  4
    /   \                     val:  5  8  7 20 15
  (8)   (7)
  / \     \                   child(i) = 2i+1, 2i+2
(20)(15)                      parent(i) = (i-1)//2</pre>
<table>
<tr><th>Op</th><th>Method</th><th>Best</th><th>Avg</th><th>Worst</th><th>Space</th></tr>
<tr><td>Peek min</td><td>h[0]</td><td>O(1)</td><td>O(1)</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Push</td><td>heappush</td><td>O(1)</td><td>O(log n)</td><td>O(log n)</td><td>O(1)</td></tr>
<tr><td>Pop min</td><td>heappop</td><td>O(log n)</td><td>O(log n)</td><td>O(log n)</td><td>O(1)</td></tr>
<tr><td>Build heap</td><td>heapify</td><td>O(n)</td><td>O(n)</td><td>O(n)</td><td>O(1)</td></tr>
<tr><td>Search arbitrary</td><td>-</td><td>O(1)</td><td>O(n)</td><td>O(n)</td><td>O(1)</td></tr>
</table>
<strong>PITFALLS</strong>
<ul class="pitfalls">
<li>Python's <code>heapq</code> is min-heap only — negate values or wrap in a comparator class for max-heap behavior.</li>
<li>Tuple comparison for ties falls through to comparing the 2nd element (payload) — can crash if payloads are incomparable (e.g., dicts); add a unique tiebreaker index.</li>
<li>A heap is <em>not</em> fully sorted — only the root is guaranteed min; don't iterate the array expecting sorted order.</li>
<li>Decreasing a key in-place (needed for Dijkstra) isn't natively supported — requires lazy deletion or an indexed heap.</li>
</ul>

In [ ]:
import heapq
h = [5, 8, 7, 20, 15]
heapq.heapify(h)                     # O(n) in-place
print('Min-heap:', h)

heapq.heappush(h, 3)
smallest = heapq.heappop(h)
print('Smallest element:', smallest)

# Max-heap emulation
neg_max_heap = [-x for x in [5, 8, 7, 20, 15]]
heapq.heapify(neg_max_heap)
largest = -heapq.heappop(neg_max_heap)
print('Largest element:', largest)

## Graphs (Adjacency List & Matrix)

<strong>ADT:</strong> set of vertices V and edges E modeling pairwise relationships. <br><strong>Impl:</strong> Adjacency List = dict/array of neighbor lists (sparse-friendly); Adjacency Matrix = V×V grid of edge weights/flags (dense-friendly, O(1) edge lookup).

<pre class="diagram">
Graph: A-B, A-C, B-C      Adjacency List      Adjacency Matrix
                               A: [B, C]           A  B  C
   (A)---(B)                   B: [A, C]        A [ 0, 1, 1 ]
     \   /                     C: [A, B]        B [ 1, 0, 1 ]
      (C)                                       C [ 1, 1, 0 ]</pre>
<table>
<tr><th>Op</th><th>List</th><th>Matrix</th></tr>
<tr><td>Add vertex</td><td>O(1)</td><td>O(V²)</td></tr>
<tr><td>Add edge</td><td>O(1)</td><td>O(1)</td></tr>
<tr><td>Remove edge</td><td>O(deg)</td><td>O(1)</td></tr>
<tr><td>Edge query (u,v)?</td><td>O(deg)</td><td>O(1)</td></tr>
<tr><td>Iterate neighbors</td><td>O(deg)</td><td>O(V)</td></tr>
<tr><td>Space</td><td>O(V+E)</td><td>O(V²)</td></tr>
<tr><td>BFS / DFS</td><td>O(V+E)</td><td>O(V²)</td></tr>
</table>
<div style="font-size:0.75rem; color:var(--text-secondary); margin-bottom: 0.5rem;">(V=VERTICES, E=EDGES)</div>
<strong>PITFALLS</strong>
<ul class="pitfalls">
<li>Adjacency matrix wastes O(V²) memory on sparse graphs — infeasible past ~10⁴ vertices; prefer list.</li>
<li>Undirected graphs need edges added <em>both</em> directions — a common off-by-one/one-way-edge bug.</li>
<li>Forgetting a <code>visited</code> set in DFS/BFS on a cyclic graph causes infinite loops/stack overflow.</li>
<li>Dense weighted graphs (Floyd-Warshall) favor matrix; sparse shortest-path (Dijkstra) favors list + heap — pick per access pattern, not habit.</li>
</ul>

In [ ]:
from collections import defaultdict

# Adjacency List
adj_list = defaultdict(list)
adj_list["A"].extend(["B", "C"])
adj_list["B"].append("A")
print('Adjacency List:', dict(adj_list))

# Adjacency Matrix
n = 3
adj_matrix = [[0]*n for _ in range(n)]
adj_matrix[0][1] = adj_matrix[1][0] = 1   # edge A-B
print('Adjacency Matrix:')
for row in adj_matrix:
    print(row)